### Logistic Regression

* Use a **logistic function** to transform distances in probability. 
* The distances are computed from the line or hyperplane, built as a weighted combination of the features.

**Logistic vs LDA**

Very similar to Linear Discriminant Analis algorithm since both compute a linear combination of the features. The difference is in what has to be optimize.\
LDA since it's a density estimator has a maximium likelihood to minimize, Logistic Regression minimize classification error.\
Also logistic is resistent to outliers since it's not interested in the distribution (it's not generative it's discriminative) while LDA assume gauss pdf so is sensible to outliers.

### Support Vector Mchine

* To decide decision boundary. Maximize the highway, find an hyperplane that separate the classes. The width is called **margin**.\
* Datapoints fixing the margin are called **support vectors**. 
* Extremly **resistent to outliers**.

In sklearn is **SVC Support Vector Classifier**.

**Multiple Support Vector**
* Another *flavor* of such method can support multiple Support vectors and admit a finite number of points within the highway.\
* The loss function minimize the number of such points and maximize the width of the highway using the smaller amout of support vectors.\
* It uses a **regularization parameter C**, acts as a L2 penalty. Quantifies confusion between clusters. We see that C controls the number of "margin violations" that are allowed, and the width of the "street" that runs between the data (i.e., the number of support vectors).
* Necessary preprocessing to work.
* Note that SVM only depends on the support vectors. Smaller C causes there to be more support vectors, while larger C gives fewer.
* It's ideal to **unbalanced classification problems** for which one class is much less present then the other(rare sources). Or for problems with lot of outliers.

## Decision Making and Random Forest

### Decision Trees

Analogia indovina chi, chiaro.

* Hyperparameter principale **depth**, numero di layer, create maximum 2^depth number of patches (the **leafs** of the tree) in which the dataset is divided at the end of the algorithm.
* How to divide the tree based is based on features splitting that is decided in the training part.
* After having the patches, use training set to check your results. Validation set: a new galaxy follow the questioning process of the tre and end up in one patch. The galaxy will be classified as the same type of the majority of galaxy train used that end up in that patch.


* Finally the optimizer i think that optimize the splitting process to get best accuracy. 
* Optimizer measures the amount of information, *entropy criterion*, to decide which features to slice at which point.\
The goal is to increase the **information gain** = reduction in entropy after partitioning the dataset. 
* Option for the optimizer are: Kullback-Leibler Divergence ('entropy'). Another possibility is 'gini'.


## Ensamble learning

Train many weak classifier, put them togheter and let them vote on the class of a new object. Majority wins. Democratic classifier.\
Each time we need to change the training set: in bagging we do it by bootstrapping, in random forest by changing the features.

### Bagging

* Works for any **weak classifier** (not only trees) and also for regression algorithms.
* This is **Bootstrap Aggregation**. In short, bagging averages the predictive results of a series of bootstrap samples of the original data. Estimator averages over all votes. All votes count the same.
* Basically define a strong classifier with as first object a weak classifier that will be repeated n times. One intersting opion in sklearn is **n_jobs=-1**, cores works in parallel, since each process is indipendent.

### Random Forest

* Identical to Bagging but specific for trees. More powerful then Bagging.
* Random forests extend bagging. In addition to drawing random samples from our training set with replacement, we may also draw random subsets of features for training the individual trees.\
* The result is to force the algorithm to move differently (by **hiding features**! since we are selecting only some of them) from decision trees -> better explore the parameter space, better minimize the loss function. Hiding features in neural network will become dropout.

### Boosting

* Loss of democracy in voting. Weights depend on how incorrectly the data were classified in the previous iteration. This is done weighting classifier with more interation (more accurate) more. 
* Pro to detect class within class, subregions. Cons it's not parallel like baggin or forest anymore, huge computational cost, tree in sequence not in parallel. 

# Can a computer learn if we're going to detect gravitational waves?

* Mtot and z will be the dominant features while ra, dec, psi, iota have non linear dep.
* Random Forest, robust choice. Because it builds hundreds of independent deep trees and averages them, it naturally handles non-linear interactions (like ra interacting with dec) and is very resistant to overfitting. Great baseline for completeness and contamination.

In [28]:
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

path = "sample_2e7_design_precessing_higherordermodes_3detectors.h5"
with h5py.File(path, 'r') as f:
    fin = {key: f[key][()] for key in f.keys()}

N_lim = 500000
data = {k: v[:N_lim] for k, v in fin.items()}
df = pd.DataFrame(data)

#selezione features
y = df['det']
X = df.drop(columns=['snr', 'det'])

print(f"\nDataset creato! Dimensioni X: {X.shape}")
print(f"Feature mantenute: {list(X.columns)}")


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\nDimensioni Training set: {X_train.shape}")
print(f"Dimensioni Validation set: {X_val.shape}")


Dataset creato! Dimensioni X: (500000, 13)
Feature mantenute: ['chi1x', 'chi1y', 'chi1z', 'chi2x', 'chi2y', 'chi2z', 'dec', 'iota', 'mtot', 'psi', 'q', 'ra', 'z']

Dimensioni Training set: (350000, 13)
Dimensioni Validation set: (150000, 13)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score
import time

rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, criterion='entropy', n_jobs=-1, random_state=42)

print("Inizio addestramento della Foresta")
start_time = time.time()
rf_clf.fit(X_train, y_train)

print(f"Tempo addestramento {time.time() - start_time:.2f} secondi!\n")

y_pred = rf_clf.predict(X_val)

# calcolo metriche solite
completeness = recall_score(y_val, y_pred)
contamination = 1 - precision_score(y_val, y_pred)

print(f"--- Risultati Baseline Random Forest ---")
print(f"Completeness (Recall):  {completeness:.4f}")
print(f"Contamination:          {contamination:.4f}")

Inizio addestramento della Foresta
Tempo addestramento 52.11 secondi!

--- Risultati Baseline Random Forest ---
Completeness (Recall):  0.8808
Contamination:          0.0856


In [30]:
# alzo la soglia dello score
predictor_soglia = rf_clf.predict_proba(X_val)[:, 1] 
y_pred_soglia = (predictor_soglia > 0.80).astype(int)

completeness = recall_score(y_val, y_pred_soglia)
contamination = 1 - precision_score(y_val, y_pred_soglia)

print(f"--- Risultati Baseline Random Forest with Threshold 0.80---")
print(f"Completeness (Recall):  {completeness:.4f}")
print(f"Contamination:          {contamination:.4f}")

--- Risultati Baseline Random Forest with Threshold 0.80---
Completeness (Recall):  0.6662
Contamination:          0.0165


In [31]:
from sklearn.metrics import confusion_matrix, accuracy_score
print("Matrice di Confusione:")
# [Veri Negativi,  Falsi Positivi]
# [Falsi Negativi, Veri Positivi]
print(confusion_matrix(y_val, y_pred))
acc = accuracy_score(y_val, y_pred)
print(f"\nAccuracy Globale: {acc:.4f}")

Matrice di Confusione:
[[126437   1794]
 [  2594  19175]]

Accuracy Globale: 0.9707


In [32]:
print("Matrice di Confusione (Soglia 0.80):")
# [Veri Negativi,  Falsi Positivi]
# [Falsi Negativi, Veri Positivi]
print(confusion_matrix(y_val, y_pred_soglia))
acc = accuracy_score(y_val, y_pred_soglia)
print(f"\nAccuracy Globale: {acc:.4f}")

Matrice di Confusione (Soglia 0.80):
[[127988    243]
 [  7267  14502]]

Accuracy Globale: 0.9499


In [ ]:
import pandas as pd
importances = pd.Series(rf_clf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)
print("--- Importanza delle Feature Fisiche ---")
print(importances.head(13))

[0.0147544  0.01487844 0.02531102 0.0141879  0.0143819  0.01617211
 0.01866051 0.05470157 0.06133666 0.01432261 0.03085859 0.02105311
 0.69938119]
--- Importanza delle Feature Fisiche ---
z        0.699381
mtot     0.061337
iota     0.054702
q        0.030859
chi1z    0.025311
ra       0.021053
dec      0.018661
chi2z    0.016172
chi1y    0.014878
chi1x    0.014754
chi2y    0.014382
psi      0.014323
chi2x    0.014188
dtype: float64
